# 02 · Variables disponibles y preparación

Comprueba la preparación real conservada en src/pipeline.py, sus dos variables derivadas y el aislamiento por persona. Se ajustan transformaciones; no clasificadores.

**Alcance de validación.** 2014J ya se había observado en una versión anterior. **No es una prueba final independiente ni un hold-out de un solo uso.** Separar personas y ajustar transformaciones dentro de cada fold evita fugas directas, pero no elimina el conocimiento previo de sus resultados. Las cifras sobre 2014J describen este histórico; la decisión institucional necesita una cohorte futura sin consultar, con protocolo fijado antes de conocer resultados.

In [1]:
from pathlib import Path
import os, sys, json, hashlib
here = Path.cwd().resolve()
ROOT = next((p for p in [here, *here.parents] if (p/'src/notebooks.py').is_file()), None)
if ROOT is None and (here/'proyecto_final/src/notebooks.py').is_file():
    ROOT = here/'proyecto_final'
assert ROOT is not None, 'Abrir desde proyecto_final o notebooks.'
sys.path.insert(0, str(ROOT))
os.environ['MPLCONFIGDIR'] = str(ROOT/'.local/matplotlib')
os.environ['PSET4_MODEL_MODE'] = 'fixture'
from IPython.display import display, Markdown
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.notebooks import evidencia
from src.datos import sha256
E = evidencia()
pd.set_option('display.max_columns', 12)
pd.set_option('display.max_rows', 32)
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})
%matplotlib inline
print('Ejecución local de evidencia congelada. Sin llamadas LLM, nuevos entrenamientos ni cambios en Registry.')


Ejecución local de evidencia congelada. Sin llamadas LLM, nuevos entrenamientos ni cambios en Registry.


## Variables y disponibilidad

La información estática se conoce al inicio; la longitudinal solo incorpora actividad hasta el corte. El resultado final, el retiro posterior y los identificadores se excluyen de los predictores.

In [2]:
from src.pipeline import columnas, construir_preparacion, FeaturesAlCorte
from src.notebooks import desarrollo
import pickle
train, snapshot, split = desarrollo()
ref = json.loads((ROOT/'tests/fixtures/referencia.json').read_text())
assert sha256(ROOT/'src/pipeline.py') == ref['pipeline_sha256']
display(pd.DataFrame([{'tipo':'estático' if static else 'estático + longitudinal','numericas':', '.join(columnas(static)[0]),'categoricas':', '.join(columnas(static)[1])} for static in [True,False]]))
transformed = FeaturesAlCorte().fit_transform(train)
display(transformed[['log1p_clicks_current_14d','clicks_per_active_day_14d']].describe())
assert transformed.clicks_per_active_day_14d.isna().sum() == train.active_days_current_14d.eq(0).sum()
print('Una intensidad sin días activos queda indefinida; la mediana se aprende solo en entrenamiento.')


,tipo,numericas,categoricas
0,estático,"num_of_prev_attempts, studied_credits, module_...","gender, region, highest_education, imd_band, a..."
1,estático + longitudinal,"num_of_prev_attempts, studied_credits, module_...","gender, region, highest_education, imd_band, a..."


,log1p_clicks_current_14d,clicks_per_active_day_14d
count,16331.000000,13729.000000
mean,3.466640,18.319299
std,1.918545,18.200391
min,0.000000,1.000000
25%,2.397895,8.666667
50%,3.988984,14.500000
75%,4.882802,23.333333
max,8.857800,1171.333333


Una intensidad sin días activos queda indefinida; la mediana se aprende solo en entrenamiento.


## Cinco folds y tres preparaciones

Se verifica que ninguna persona aparezca en ambos lados, las medianas se aprendan del entrenamiento y transformar la validación no reajuste estadísticas. Alterar etiquetas o columnas futuras ajenas a la lista de predictores debe dejar la salida intacta.

In [3]:
checks=[]
for name, options in [('estatica',{'solo_estaticas':True}),('longitudinal',{'derivadas':False}),('longitudinal_derivadas',{})]:
    for fold in sorted(train.cv_fold.unique()):
        fit=train.loc[train.cv_fold.ne(fold)]; valid=train.loc[train.cv_fold.eq(fold)]
        assert not set(fit.id_student) & set(valid.id_student)
        pipeline=construir_preparacion(**options).fit(fit)
        state=pickle.dumps(pipeline); z=pipeline.transform(valid)
        assert np.isfinite(z).all() and pickle.dumps(pipeline)==state
        numeric,_=columnas(**options)
        numeric_train=pipeline.named_steps['features'].transform(fit)[numeric]
        expected=np.array([col.median() if col.notna().any() else 0 for _,col in numeric_train.items()])
        np.testing.assert_allclose(pipeline.named_steps['preparacion'].named_transformers_['num'].named_steps['imputar'].statistics_,expected)
        changed=valid.assign(target_class='Approved',date_unregistration=-999,id_student=-1,future_clicks=10**9)
        np.testing.assert_array_equal(z,pipeline.transform(changed))
        assert pickle.dumps(pipeline)==state
        checks.append({'variante':name,'fold':int(fold),'filas_fit':len(fit),'filas_validacion':len(valid),'personas_compartidas':0,'salida_finita':True,'medianas_solo_fit':True})
display(pd.DataFrame(checks))
assert len(checks)==15
print('15 preparaciones verificadas; pipeline conservado; cero clasificadores ajustados.')


,variante,fold,filas_fit,filas_validacion,personas_compartidas,salida_finita,medianas_solo_fit
0,estatica,0,13055,3276,0,True,True
1,estatica,1,13049,3282,0,True,True
2,estatica,2,13066,3265,0,True,True
3,estatica,3,13054,3277,0,True,True
4,estatica,4,13100,3231,0,True,True
5,longitudinal,0,13055,3276,0,True,True
6,longitudinal,1,13049,3282,0,True,True
7,longitudinal,2,13066,3265,0,True,True
8,longitudinal,3,13054,3277,0,True,True
9,longitudinal,4,13100,3231,0,True,True


15 preparaciones verificadas; pipeline conservado; cero clasificadores ajustados.


## Lectura final

Los resultados reproducidos pertenecen a los experimentos registrados. La fecha de ejecución de este notebook no los convierte en evidencia nueva de generalización. Consulte `README.md` para insumos, dependencias y comandos del pipeline completo.